In [1]:
# Import the necessary libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset

In [2]:
# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

In [3]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Define column types based on provided metadata
continuous_columns = [
    'dog', 'panda', 'squirrel', 'dolphin', 'penguin', 'turtle', 'elephant', 'lamb', 'cow', 'horse',
    'chicken', 'sparrow', 'finch', 'bee', 'ladybug', 'cricket', 'zebra', 'flamingo', 'peacock', 'bat',
    'fox', 'beaver', 'seal', 'robin', 'swan', 'minnow', 'mole', 'owl', 'bunny', 'bear', 'cloud',
    'rainbow', 'puddle', 'apple', 'honey', 'button', 'whistle', 'marble', 'bubble'
]  # TABULAR_NUMERIC_DISCRETE
continuous_columns += [
    'rabbit', 'deer', 'hedgehog', 'mouse', 'guinea', 'parrot', 'canary', 'wombat', 'loon', 'goldfish',
    'puffin', 'cub', 'acorn', 'berry', 'pumpkin', 'wagon', 'storybook', 'clover', 'frog'
]  # TABULAR_NUMERIC_DIGIT
continuous_columns += ['snail', 'shrew']  # TABULAR_NUMERIC_BINNED

categorical_columns = [
    'cat', 'koala', 'otter', 'giraffe', 'goat', 'monkey', 'donkey', 'pony', 'llama', 'hamster', 'duck',
    'butterfly', 'tamarin', 'wallaby', 'chipmunk', 'leaf', 'teddy', 'blanket', 'candle', 'cookie'
]  # TABULAR_CATEGORICAL

Using device: cpu


In [4]:
# Load dataset
df = pd.read_csv(r"C:\Users\Abdulrahman A\mostly_flat_challenge\datasets\flat-training (original).csv")
print(f"Dataset shape: {df.shape}")  # Should be (100000, 81)
print(df.head())

# Save original categorical columns for evaluation
df_original = df[categorical_columns].copy().astype(str)

Dataset shape: (100000, 80)
   dog   cat  rabbit  deer  panda koala otter  hedgehog  squirrel  dolphin  \
0   10  A5DB     NaN  4.46     -2    T2  B9DE      51.8         0        1   
1   10  A5DB     NaN  4.42      0    T3  027A      72.2         1        1   
2   43  027A     8.0  3.11     -1    T0  B9DE      44.2         1        1   
3   28  63D1     NaN  3.37     -1    T1  027A      41.0         0        1   
4   82  C09E     NaN  3.07     -6    T0  B9DE      46.4         1        1   

   ...  blanket  button  whistle marble  wagon storybook  candle  clover  \
0  ...       A8       2        2      0  -0.76       -54      B2    0.38   
1  ...       A7      16        0      0  -0.76       -48      B0    0.16   
2  ...       A4     -25       10      0  -0.88       -39      B0    0.13   
3  ...       A3      21        9      0  -0.85       -58      B1    0.29   
4  ...       A7      -4       13      0  -0.85        70      B2    0.75   

  bubble cookie  
0      0    C13  
1      0  

In [5]:
# Checking out each columns for complete or missing values
df.info()

# Output: Rabbit has only 24024 datasets, Cow has 50003, and Cloud has 20596

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 80 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   dog        100000 non-null  int64  
 1   cat        100000 non-null  object 
 2   rabbit     24024 non-null   float64
 3   deer       100000 non-null  float64
 4   panda      100000 non-null  int64  
 5   koala      100000 non-null  object 
 6   otter      100000 non-null  object 
 7   hedgehog   100000 non-null  float64
 8   squirrel   100000 non-null  int64  
 9   dolphin    100000 non-null  int64  
 10  penguin    100000 non-null  int64  
 11  turtle     100000 non-null  float64
 12  elephant   100000 non-null  int64  
 13  giraffe    100000 non-null  object 
 14  lamb       100000 non-null  int64  
 15  goat       100000 non-null  object 
 16  cow        50003 non-null   float64
 17  horse      100000 non-null  int64  
 18  donkey     100000 non-null  object 
 19  pony       100000 non-nu

In [6]:
# Finding columns with missing numbers
miss_df = df.isnull().sum()
print(miss_df)

dog              0
cat              0
rabbit       75976
deer             0
panda            0
             ...  
storybook        0
candle           0
clover           0
bubble           0
cookie           0
Length: 80, dtype: int64


In [7]:
# Handle missing values
# Continuous columns: Impute missing values for rabbit, cow, cloud specifically
# Convert to numeric, handling non-numeric values
df[continuous_columns] = df[continuous_columns].apply(pd.to_numeric, errors='coerce')

# Impute missing values for rabbit, cow, cloud with their respective means, rounded and clipped
for column in ['rabbit', 'cow', 'cloud']:
    if column in continuous_columns:
        mean_value = df[column].mean()  # Compute mean of non-null values
        rounded_mean = np.round(mean_value).astype(int)  # Round to nearest integer
        # Clip to specific ranges for rabbit and cloud
        if column == 'rabbit':
            rounded_mean = np.clip(rounded_mean, 0, 100)
        elif column == 'cloud':
            rounded_mean = np.clip(rounded_mean, -10, 10)
        df[column] = df[column].fillna(rounded_mean)
        print(f"Imputed {df[column].isna().sum()} missing values in {column} with rounded mean {rounded_mean}")

# Impute other continuous columns (if any have missing values) with mean
other_continuous = [col for col in continuous_columns if col not in ['rabbit', 'cow', 'cloud']]
df[other_continuous] = df[other_continuous].fillna(df[other_continuous].mean())

# Verify missing values are handled
print("\nMissing values per column after imputation:")
print(df[['rabbit', 'cow', 'cloud']].isna().sum())

Imputed 0 missing values in rabbit with rounded mean 39
Imputed 0 missing values in cow with rounded mean 1
Imputed 0 missing values in cloud with rounded mean 0

Missing values per column after imputation:
rabbit    0
cow       0
cloud     0
dtype: int64


In [8]:
print(df[['rabbit', 'cow', 'cloud']].describe())

              rabbit            cow          cloud
count  100000.000000  100000.000000  100000.000000
mean       39.025080       0.823233       0.040840
std        13.496793       0.227238       2.632501
min         1.000000       0.430000     -10.000000
25%        39.000000       0.580000       0.000000
50%        39.000000       1.000000       0.000000
75%        39.000000       1.000000       0.000000
max       100.000000       1.250000      10.000000


In [9]:
# Check the info() to confirm all the datasets are of the same structures
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 80 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   dog        100000 non-null  int64  
 1   cat        100000 non-null  object 
 2   rabbit     100000 non-null  float64
 3   deer       100000 non-null  float64
 4   panda      100000 non-null  int64  
 5   koala      100000 non-null  object 
 6   otter      100000 non-null  object 
 7   hedgehog   100000 non-null  float64
 8   squirrel   100000 non-null  int64  
 9   dolphin    100000 non-null  int64  
 10  penguin    100000 non-null  int64  
 11  turtle     100000 non-null  float64
 12  elephant   100000 non-null  int64  
 13  giraffe    100000 non-null  object 
 14  lamb       100000 non-null  int64  
 15  goat       100000 non-null  object 
 16  cow        100000 non-null  float64
 17  horse      100000 non-null  int64  
 18  donkey     100000 non-null  object 
 19  pony       100000 non-nu

In [10]:
# Check the dataset
print("The first five rows:")
print(df.head())
print("\n The last five rows:")
print(df.tail())

The first five rows:
   dog   cat  rabbit  deer  panda koala otter  hedgehog  squirrel  dolphin  \
0   10  A5DB    39.0  4.46     -2    T2  B9DE      51.8         0        1   
1   10  A5DB    39.0  4.42      0    T3  027A      72.2         1        1   
2   43  027A     8.0  3.11     -1    T0  B9DE      44.2         1        1   
3   28  63D1    39.0  3.37     -1    T1  027A      41.0         0        1   
4   82  C09E    39.0  3.07     -6    T0  B9DE      46.4         1        1   

   ...  blanket  button  whistle marble  wagon storybook  candle  clover  \
0  ...       A8       2        2      0  -0.76       -54      B2    0.38   
1  ...       A7      16        0      0  -0.76       -48      B0    0.16   
2  ...       A4     -25       10      0  -0.88       -39      B0    0.13   
3  ...       A3      21        9      0  -0.85       -58      B1    0.29   
4  ...       A7      -4       13      0  -0.85        70      B2    0.75   

  bubble cookie  
0      0    C13  
1      0    C15  

In [11]:
# Encode categorical columns with LabelEncoder (for conditional vector)
label_encoders = {}
for column in categorical_columns:
    le = LabelEncoder()
    df[f"{column}_encoded"] = le.fit_transform(df[column])
    label_encoders[column] = le
    print(f"Encoded {column} with classes: {le.classes_}")

# One-hot encode categorical columns
one_hot_encoders = {}
ohe_columns = []
for column in categorical_columns:
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    encoded = ohe.fit_transform(df[[column]])
    ohe_col_names = [f"{column}_{i}" for i in range(encoded.shape[1])]
    encoded_df = pd.DataFrame(encoded, columns=ohe_col_names)
    df = pd.concat([df, encoded_df], axis=1)
    one_hot_encoders[column] = ohe
    ohe_columns.extend(ohe_col_names)

# Drop original and encoded categorical columns
df = df.drop(categorical_columns + [f"{col}_encoded" for col in categorical_columns], axis=1)

# Debug: Verify no object dtypes remain
print("\nColumns with object/string dtypes after encoding:")
object_cols = df.select_dtypes(include=['object', 'string']).columns
if not object_cols.empty:
    print(object_cols)
    for col in object_cols:
        print(f"Sample values for {col}: {df[col].unique()[:5]}")
else:
    print("No object/string columns remain.")

# Normalize continuous columns using StandardScaler and adaptive GMM
scalers = {}
gmm_models = {}
for column in continuous_columns:
    scaler = StandardScaler()
    df[[column]] = scaler.fit_transform(df[[column]])
    scalers[column] = scaler
    n_unique = len(df[column].unique())
    n_components = min(n_unique, 5)
    if n_components > 1:
        gmm = GaussianMixture(n_components=n_components, random_state=42)
        gmm.fit(df[[column]])
        gmm_models[column] = gmm
    else:
        gmm_models[column] = None
        print(f"Skipped GMM for {column} (only {n_unique} unique value(s))")

# Verify dtypes to ensure all are numeric
print("\nDataFrame dtypes after preprocessing:")
print(df.dtypes)

# Ensure all columns are float32 for PyTorch compatibility
try:
    df = df.astype(np.float32)
except ValueError as e:
    print(f"Error converting to float32: {e}")
    for column in df.columns:
        if df[column].dtype not in ['float64', 'float32', 'int64', 'int32']:
            print(f"Non-numeric column {column}: dtype={df[column].dtype}, sample={df[column].head().tolist()}")
    raise

# Store all columns
all_columns = df.columns.tolist()

Encoded cat with classes: ['027A' '0D45' '1520' '18FD' '19CE' '248B' '25CD' '39ED' '457A' '580C'
 '5A6C' '63D1' '66DA' '6C1C' '7202' 'A10E' 'A2FB' 'A5DB' 'A639' 'A864'
 'ACAB' 'B9DE' 'BB39' 'C09E' 'C2D8' 'C9CF' 'D668' 'E304' 'F068']
Encoded koala with classes: ['T0' 'T1' 'T2' 'T3']
Encoded otter with classes: ['027A' 'B9DE' 'C09E']
Encoded giraffe with classes: ['n' 'y']
Encoded goat with classes: ['027A' 'B9DE']
Encoded monkey with classes: ['Z0' 'Z1' 'Z2' 'Z3']
Encoded donkey with classes: ['X0' 'X1' 'X2' 'X3' 'X4' 'X5' 'X6' 'X7' 'X8']
Encoded pony with classes: ['5' 'M']
Encoded llama with classes: ['027A' 'B9DE']
Encoded hamster with classes: ['D0' 'D1' 'D10' 'D11' 'D2' 'D3' 'D4' 'D5' 'D6' 'D7' 'D8' 'D9']
Encoded duck with classes: ['027A' '0D45' '1520' '19CE' '248B' '457A' '5A6C' '66DA' '6C1C' 'A2FB'
 'B9DE' 'BB39' 'C09E' 'C9CF']
Encoded butterfly with classes: ['027A' '0571' '0BD7' '0D45' '1520' '18FD' '19CE' '248B' '25CD' '2A4C'
 '2C78' '30AE' '391C' '39ED' '40E4' '457A' '580C' 

In [12]:
# Define the Generator
class Generator(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim * 2),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, output_dim),
            nn.Tanh()
        )
    
    def forward(self, z, cond):
        x = torch.cat([z, cond], dim=1)
        return self.model(x)

In [13]:
# Discriminator Definition
class Discriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim * 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x, cond):
        x = torch.cat([x, cond], dim=1)
        return self.model(x)

In [14]:
# Conditional vector sampling
def sample_conditional_vector(batch_size, categorical_columns, one_hot_encoders, device):
    cond_vectors = []
    for column in categorical_columns:
        n_categories = len(one_hot_encoders[column].categories_[0])
        categories = np.random.randint(0, n_categories, batch_size)
        one_hot = np.zeros((batch_size, n_categories))
        one_hot[np.arange(batch_size), categories] = 1
        cond_vectors.append(torch.tensor(one_hot, dtype=torch.float32, device=device))
    return torch.cat(cond_vectors, dim=1)

In [15]:
# Define the Hyperparameters
batch_size = 512  # Increased for 100,000 samples
n_epochs = 5
noise_dim = 128
hidden_dim = 256
lr = 0.0005
gradient_penalty_weight = 10

In [16]:
# Initialize models
cond_dim = sum(len(ohe.categories_[0]) for ohe in one_hot_encoders.values())
generator = Generator(noise_dim + cond_dim, hidden_dim, len(all_columns)).to(device)
discriminator = Discriminator(len(all_columns) + cond_dim, hidden_dim).to(device)

In [17]:
# Optimizers
optimizer_G = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.9))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.9))

In [18]:
# Loss function
criterion = nn.BCELoss()

In [19]:
# Gradient penalty for WGAN-GP
def compute_gradient_penalty(discriminator, real_samples, fake_samples, cond_vector, device):
    alpha = torch.rand(real_samples.size(0), 1).to(device)
    alpha = alpha.expand_as(real_samples)
    interpolates = alpha * real_samples + (1 - alpha) * fake_samples
    interpolates.requires_grad_(True)
    d_interpolates = discriminator(interpolates, cond_vector)
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=torch.ones_like(d_interpolates).to(device),
        create_graph=True,
        retain_graph=True
    )[0]
    gradients = gradients.view(gradients.size(0), -1)
    gradient_norm = gradients.norm(2, dim=1)
    return ((gradient_norm - 1) ** 2).mean()


In [23]:
# Data loader
data_tensor = torch.tensor(df.values, dtype=torch.float32).to(device)
data_loader = DataLoader(data_tensor, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)

In [24]:
# Training loop
for epoch in range(n_epochs):
    for i, real_data in enumerate(data_loader):
        batch_size_i = real_data.size(0)
        
        # Labels
        real_labels = torch.ones(batch_size_i, 1).to(device)
        fake_labels = torch.zeros(batch_size_i, 1).to(device)
        
        # Sample conditional vector
        cond_vector = sample_conditional_vector(batch_size_i, categorical_columns, one_hot_encoders, device)
        
        # Train Discriminator
        optimizer_D.zero_grad()
        
        # Real data
        d_real = discriminator(real_data, cond_vector)
        loss_d_real = criterion(d_real, real_labels)
        
        # Fake data
        noise = torch.randn(batch_size_i, noise_dim).to(device)
        fake_data = generator(noise, cond_vector)
        d_fake = discriminator(fake_data.detach(), cond_vector)
        loss_d_fake = criterion(d_fake, fake_labels)
        
        # Gradient penalty
        gradient_penalty = compute_gradient_penalty(discriminator, real_data, fake_data.detach(), cond_vector, device)
        
        # Total discriminator loss
        loss_d = loss_d_real + loss_d_fake + gradient_penalty_weight * gradient_penalty
        loss_d.backward()
        optimizer_D.step()
        
        # Train Generator
        optimizer_G.zero_grad()
        d_fake = discriminator(fake_data, cond_vector)
        loss_g = criterion(d_fake, real_labels)
        loss_g.backward()
        optimizer_G.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{n_epochs}] Loss D: {loss_d.item():.4f}, Loss G: {loss_g.item():.4f}")

RuntimeError: DataLoader worker (pid(s) 16964, 18832, 5236, 14676) exited unexpectedly

In [ ]:
# Generate synthetic data
n_synthetic = 100000
with torch.no_grad():
    cond_vector = sample_conditional_vector(n_synthetic, categorical_columns, one_hot_encoders, device)
    noise = torch.randn(n_synthetic, noise_dim).to(device)
    synthetic_data = generator(noise, cond_vector).cpu().numpy()

# Convert to DataFrame
synthetic_df = pd.DataFrame(synthetic_data, columns=all_columns)

In [ ]:
# Decode one-hot encoded categorical columns
for column in categorical_columns:
    ohe_columns = [col for col in synthetic_df.columns if col.startswith(f"{column}_")]
    ohe_data = synthetic_df[ohe_columns].values
    # Ensure ohe_data is numeric and handle any non-numeric values
    ohe_data = np.nan_to_num(ohe_data, nan=0.0)  # Replace NaN with 0
    decoded = one_hot_encoders[column].inverse_transform(ohe_data)
    synthetic_df[column] = decoded.flatten()
    synthetic_df = synthetic_df.drop(ohe_columns, axis=1)

In [ ]:
# Denormalize continuous columns and apply constraints
for column in continuous_columns:
    synthetic_df[column] = scalers[column].inverse_transform(synthetic_df[[column]])
    # Round to integers for NUMERIC_DIGIT and NUMERIC_DISCRETE
    if column in [
        'rabbit', 'deer', 'hedgehog', 'mouse', 'guinea', 'parrot', 'canary', 'wombat',
        'loon', 'goldfish', 'puffin', 'cub', 'acorn', 'berry', 'pumpkin', 'wagon', 'storybook', 'clover', 'frog'
    ] + [
        'dog', 'panda', 'squirrel', 'dolphin', 'penguin', 'turtle', 'elephant', 'lamb', 'cow', 'horse',
        'chicken', 'sparrow', 'finch', 'bee', 'ladybug', 'cricket', 'zebra', 'flamingo', 'peacock', 'bat',
        'fox', 'beaver', 'seal', 'robin', 'swan', 'minnow', 'mole', 'owl', 'bunny', 'bear', 'cloud',
        'rainbow', 'puddle', 'apple', 'honey', 'button', 'whistle', 'marble', 'bubble', 'monkey'
    ]:
        synthetic_df[column] = np.round(synthetic_df[column]).astype(int)
        # Apply range constraints for rabbit and cloud
        if column == 'rabbit':
            synthetic_df[column] = np.clip(synthetic_df[column], 0, 100)
        elif column == 'cloud':
            synthetic_df[column] = np.clip(synthetic_df[column], -10, 10)

In [ ]:
# # Decode categorical columns back to original labels
# for column in categorical_columns:
#     synthetic_df[column] = label_encoders[column].inverse_transform(synthetic_df[column].astype(int))

In [ ]:
# Save synthetic data
synthetic_df.to_csv('synthetic_data.csv', index=False)
print("\nSynthetic dataset saved as 'synthetic_data.csv'")
print(synthetic_df.head())

In [ ]:
# Evaluate synthetic data
print("\nEvaluating synthetic data quality...")

# Split real data into training and holdout sets
train_df, holdout_df = train_test_split(df_original.join(df), test_size=0.2, random_state=42)
print(f"Training set shape: {train_df.shape}, Holdout set shape: {holdout_df.shape}")

# DCR Share (optimized with batch processing)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_df[continuous_columns]).astype(np.float32)
X_holdout_scaled = scaler.transform(holdout_df[continuous_columns]).astype(np.float32)
X_synth_scaled = scaler.transform(synthetic_df[continuous_columns]).astype(np.float32)

batch_size = 500
n_synth = X_synth_scaled.shape[0]
closer_to_train = 0

nn_train = NearestNeighbors(n_neighbors=1, n_jobs=-1)
nn_train.fit(X_train_scaled)
nn_holdout = NearestNeighbors(n_neighbors=1, n_jobs=-1)
nn_holdout.fit(X_holdout_scaled)

for start_idx in range(0, n_synth, batch_size):
    end_idx = min(start_idx + batch_size, n_synth)
    X_batch = X_synth_scaled[start_idx:end_idx]
    distances_train, _ = nn_train.kneighbors(X_batch)
    distances_holdout, _ = nn_holdout.kneighbors(X_batch)
    closer_to_train += np.sum(distances_train.flatten() < distances_holdout.flatten())

dcr_share = closer_to_train / n_synth
print(f"DCR Share: {dcr_share:.4f}")

In [ ]:

# Overall Accuracy (L1 distance on discretized marginals)
n_bins = 10
l1_distances = []

# 1D Marginals
for column in continuous_columns + categorical_columns:
    if column in continuous_columns:
        if column not in synthetic_df.columns or column not in df.columns:
            print(f"Skipping {column}: not found in df or synthetic_df")
            continue
        real_data = df[column].dropna()
        synth_data = synthetic_df[column].dropna()
        if len(real_data) == 0 or len(synth_data) == 0:
            print(f"Skipping {column}: empty after dropna")
            continue
        if len(real_data.unique()) <= 1 or len(synth_data.unique()) <= 1:
            print(f"Skipping {column}: constant value")
            continue
        bins = np.histogram_bin_edges(real_data, bins=n_bins)
        real_hist, _ = np.histogram(real_data, bins=bins, density=True)
        synth_hist, _ = np.histogram(synth_data, bins=bins, density=True)
        real_hist = np.nan_to_num(real_hist, nan=0.0)
        synth_hist = np.nan_to_num(synth_hist, nan=0.0)
    else:
        if column not in synthetic_df.columns or column not in df_original.columns:
            print(f"Skipping {column}: not found in df_original or synthetic_df")
            continue
        real_counts = df_original[column].value_counts(normalize=True)
        synth_counts = synthetic_df[column].value_counts(normalize=True)
        all_categories = sorted(set(real_counts.index).union(set(synth_counts.index)))
        real_hist = [real_counts.get(cat, 0) for cat in all_categories]
        synth_hist = [synth_counts.get(cat, 0) for cat in all_categories]
    l1_distance = np.mean(np.abs(np.array(real_hist) - np.array(synth_hist)))
    l1_distances.append(l1_distance)

# 2D Marginals (key columns)
key_columns = ['rabbit', 'cow', 'cloud', 'cat', 'monkey']
for col1, col2 in combinations(key_columns, 2):
    if col1 in continuous_columns and col2 in continuous_columns:
        if col1 not in synthetic_df.columns or col2 not in synthetic_df.columns:
            print(f"Skipping {col1} vs {col2}: not found in synthetic_df")
            continue
        real_data = df[[col1, col2]].dropna()
        synth_data = synthetic_df[[col1, col2]].dropna()
        if len(real_data) == 0 or len(synth_data) == 0:
            print(f"Skipping {col1} vs {col2}: empty after dropna")
            continue
        bins = [np.histogram_bin_edges(df[col], bins=n_bins) for col in [col1, col2]]
        real_hist, _, _ = np.histogram2d(real_data[col1], real_data[col2], bins=bins, density=True)
        synth_hist, _, _ = np.histogram2d(synth_data[col1], synth_data[col2], bins=bins, density=True)
        real_hist = np.nan_to_num(real_hist, nan=0.0)
        synth_hist = np.nan_to_num(synth_hist, nan=0.0)
        l1_distance = np.mean(np.abs(real_hist.flatten() - synth_hist.flatten()))
        l1_distances.append(l1_distance)

overall_accuracy = np.mean(l1_distances) if l1_distances else np.nan
print(f"Overall Accuracy (Average L1 Distance): {overall_accuracy:.4f}")

In [ ]:

# Existing Metrics
ks_results = {}
wasserstein_results = {}
jsd_results = {}
chi2_results = {}
for column in continuous_columns:
    if column not in synthetic_df.columns or column not in df.columns:
        continue
    real_data = df[column].dropna()
    synth_data = synthetic_df[column].dropna()
    if len(real_data) == 0 or len(synth_data) == 0:
        continue
    ks_stat, ks_pval = ks_2samp(real_data, synth_data)
    ks_results[column] = (ks_stat, ks_pval)
    wass_dist = wasserstein_distance(real_data, synth_data)
    wasserstein_results[column] = wass_dist
for column in categorical_columns:
    if column not in synthetic_df.columns or column not in df_original.columns:
        continue
    real_counts = df_original[column].value_counts(normalize=True)
    synth_counts = synthetic_df[column].value_counts(normalize=True)
    all_categories = sorted(set(real_counts.index).union(set(synth_counts.index)))
    real_probs = [real_counts.get(cat, 0) for cat in all_categories]
    synth_probs = [synth_counts.get(cat, 0) for cat in all_categories]
    jsd = 0.5 * (mutual_info_score(real_probs, synth_probs) + mutual_info_score(synth_probs, real_probs))
    jsd_results[column] = jsd
    contingency_table = pd.crosstab(df_original[column], synthetic_df[column])
    chi2_stat, chi2_pval, _, _ = chi2_contingency(contingency_table)
    chi2_results[column] = (chi2_stat, chi2_pval)

print("\nUnivariate Distribution Metrics:")
print("Continuous Columns (KS Test and Wasserstein Distance):")
for col in continuous_columns:
    if col in ks_results:
        print(f"{col}: KS Statistic={ks_results[col][0]:.4f}, KS p-value={ks_results[col][1]:.4f}, Wasserstein={wasserstein_results[col]:.4f}")
print("\nCategorical Columns (Jensen-Shannon Divergence and Chi-Square Test):")
for col in categorical_columns:
    if col in jsd_results:
        print(f"{col}: JSD={jsd_results[col]:.4f}, Chi2 p-value={chi2_results[col][1]:.4f}")

In [ ]:

# Correlation Distance
real_corr = df[continuous_columns].corr()
synth_corr = synthetic_df[continuous_columns].corr()
corr_distance = np.abs(real_corr - synth_corr).mean().mean()
print(f"\nCorrelation Distance (Continuous Columns): {corr_distance:.4f}")

# Visualize correlation matrices
plt.figure(figsize=(10, 8))
sns.heatmap(real_corr, cmap='coolwarm', center=0)
plt.title("Real Data Correlation Matrix")
plt.show()

plt.figure(figsize=(10, 8))
sns.heatmap(synth_corr, cmap='coolwarm', center=0)
plt.title("Synthetic Data Correlation Matrix")
plt.show()

In [ ]:

# Nearest Neighbor Distance Ratio (NNDR)
nn = NearestNeighbors(n_neighbors=2, n_jobs=-1)
nn.fit(X_train_scaled)
distances_synth = []
for start_idx in range(0, n_synth, batch_size):
    end_idx = min(start_idx + batch_size, n_synth)
    X_batch = X_synth_scaled[start_idx:end_idx]
    dist_batch, _ = nn.kneighbors(X_batch)
    distances_synth.append(dist_batch)
distances_synth = np.concatenate(distances_synth)
distances_real, _ = nn.kneighbors(X_train_scaled)
nndr = distances_synth[:, 0].mean() / distances_real[:, 1].mean()
print(f"Nearest Neighbor Distance Ratio (NNDR): {nndr:.4f}")

In [ ]:
# Visualizations
sample_columns = ['rabbit', 'cow', 'cloud', 'cat', 'monkey']
for column in sample_columns:
    plt.figure(figsize=(8, 4))
    if column in continuous_columns:
        if column not in synthetic_df.columns or column not in df.columns:
            continue
        sns.histplot(df[column], label='Real', alpha=0.5)
        sns.histplot(synthetic_df[column], label='Synthetic', alpha=0.5)
        plt.title(f"Distribution of {column}")
    else:
        if column not in synthetic_df.columns or column not in df_original.columns:
            continue
        real_counts = df_original[column].value_counts()
        synthetic_counts = synthetic_df[column].value_counts()
        pd.DataFrame({'Real': real_counts, 'Synthetic': synthetic_counts}).plot(kind='bar')
        plt.title(f"Category Distribution of {column}")
    plt.legend()
    plt.show()
